In [22]:
from pydantic import BaseModel, ValidationError, Field, EmailStr, field_validator
from datetime import date, datetime

class User(BaseModel):
    name: str = Field(min_length=2, max_length=50)
    age: int = Field(gt=0, lt=100)
    email: EmailStr = Field(description="Email stufff. You know right?")
    is_active: bool = Field(default=True)
    bio: str | None = Field(default=None)
    date_of_birth: date = Field(alias="birth_date")

    @field_validator("date_of_birth")
    @classmethod
    def check_age(cls, date_of_birth: date) -> date:
        today = date.today()
        eighteen_years_ago_date = date(today.year - 18, today.month, today.day)
        if date_of_birth > eighteen_years_ago_date:
            raise ValueError("Must be greater than 18 years old")
        return date_of_birth

user1 = {"name": "dineth", "age": 18, "email": "dil@gmail.com", "is_active": True, "birth_date": "2001-05-08"}
User.model_validate(user1)

User(name='dineth', age=18, email='dil@gmail.com', is_active=True, bio=None, date_of_birth=datetime.date(2001, 5, 8))

In [45]:
from enum import Enum
from pydantic import model_validator
from typing import Self

class Department(Enum):
    HR = "HR"
    SALES = "SALES"
    IT = "IT"

class Employee(BaseModel):
    name: str
    department: Department
    elected_benefits: bool

    @model_validator(mode="after")
    def elected_benefits_validate(self) -> Self:
        if self.department == Department.IT and self.elected_benefits:
            raise ValueError("IT department dont get elected_benefits")
        return self

e1 = Employee(
    name = "dineth",
    department = "IT",
    elected_benefits = False
)

print(e1)

name='dineth' department=<Department.IT: 'IT'> elected_benefits=False


In [54]:
class Address(BaseModel):
    city: str
    country: str

class User(BaseModel):
    name: str
    address: Address

u1 = User.model_validate({
    "name": "Dineth Wickramasinghe",
    "address": {
        "city": "Horana",
        "country": "Sri Lanka"
    }
})

print(type(u1))


<class '__main__.User'>
Horana
Horana


In [70]:
from typing import Annotated, Any
from pydantic import AfterValidator, BeforeValidator, PlainValidator, ValidationInfo, ValidatorFunctionWrapHandler, WrapValidator

def strip_whitespace(value: str) -> str:
    return value.strip()
    
def msut_be_adult(value: date) -> date:
    today_date = date.today()
    eighteen_years_ago_date = date(today_date.year - 18, today_date.month, today_date.day)
    if value > eighteen_years_ago_date:
        raise ValueError("Must be an adult")
    return value

class Employee(BaseModel):
    name: Annotated[str, BeforeValidator(strip_whitespace), Field(min_length=3, max_length=100)]
    age: Annotated[date, AfterValidator(msut_be_adult)]

e1 = Employee(name= "dineth ", age="2000-10-19")
e1.model_dump_json()

'{"name":"dineth","age":"2000-10-19"}'

In [85]:
## Exercises

class PhoneNumber(BaseModel):
    number: str

    @field_validator("number")
    @classmethod
    def steralize_number(cls, number: str) -> str:
        striped_number =  number.replace(" ", "").replace("-", "")
        if not 10 <= len(striped_number) < 15:
            raise ValueError("Number must be in between 10-15")
            
p1 = PhoneNumber(number = "1111111111111")
print(p1)

number=None


In [7]:
from pydantic import BaseModel, model_validator, Field, field_validator
from typing import Annotated, Self
from datetime import date

class CreditCard(BaseModel):
    number: str
    expiry_month: Annotated[int, Field(gt=1, lt=12)]
    expiry_year: int
    cvv: str = Field(min_length=3, max_length=4)

    @model_validator(mode = "after")
    def check_expire(self) -> Self:
        today = date.today()
        expiry_date_of_card = date(self.expiry_year, self.expiry_month, 1)
        if expiry_date_of_card > today:
            raise ValueError("Card Expired!!!")
        return self

c1 = CreditCard(number="100010100020", expiry_month=9, expiry_year=2002, cvv="333")
print(c1)

number='100010100020' expiry_month=9 expiry_year=2002 cvv='333'


In [46]:
class Order(BaseModel):
    items: list[str]
    subtotal: float = Field(gt=0)
    discount: float = Field(default=0.0, ge=0, le=100)
    total: float

    @model_validator(mode="after")
    def verify_total(self) -> Self:
        cal_total = self.subtotal*(1-self.discount/100)
        if self.total != cal_total:
            raise ValueError("Total is wrong!!")
        return self

    @field_validator("items")
    @classmethod
    def remove_duplicates(cls, items: list[str]) -> list[str]:
        items_set = set(items)
        items = list(items_set)
        return items
        
o1 = Order(items=["mouse", "bottle", "bottle"], subtotal=100, discount=10, total=90)
print(o1.model_dump_json(indent=2))


{
  "items": [
    "bottle",
    "mouse"
  ],
  "subtotal": 100.0,
  "discount": 10.0,
  "total": 90.0
}


In [5]:
from typing import Literal, Union, Annotated
from datetime import datetime

class Department(BaseModel):
    id: int
    name: str
    floor: int

class Doctor(BaseModel):
    id: int
    name: str
    spec: str
    department: Department | None = None
    exp: Annotated[int, Field(ge=0)]

    @model_validator(mode="after")
    def dep_not_none(self) -> Self:
        if self.department is None:
            raise ValueError("Department is null")
        return self

class Patient(BaseModel):
    id: int
    name: str
    dob: date
    blood_type: Literal['A+','A-','B+','B-','AB+','AB-','O+','O-']

class Appointment(BaseModel):
    id: int
    patient: Patient
    doctor: Doctor
    sche: datetime
    dura: int = Field(gt=15, lt=240)
    notes: Optional[str] = None

class Hospital(BaseModel):
    name: str
    departments: list[Department]
    doctors: list[Doctor]
    appointments: list[Appointment]

    @model_validator(mode="after")
    def verify_id(self) -> Self:
        dep_id_list = [dep.id for dep in self.departments]
        for doctor in self.doctors:
            if doctor.department.id not in dep_id_list:
                raise ValueError(f"Wrong department id in {doctor}")
        return self


ImportError: cannot import name 'Field' from 'typing' (C:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\typing.py)

In [62]:
hospital_data = {
    "name": "City General Hospital",
    "departments": [
        {
            "id": 1,
            "name": "Cardiology",
            "floor": 3
        },
        {
            "id": 2,
            "name": "Neurology",
            "floor": 5
        }
    ],
    "doctors": [
        {
            "id": 101,
            "name": "Dr. Sarah Fernando",
            "spec": "Cardiologist",
            "department": {
                "id": 1,
                "name": "Cardiology",
                "floor": 3
            },
            "exp": 12
        },
        {
            "id": 102,
            "name": "Dr. Nimal Perera",
            "spec": "Neurologist",
            "department": {
                "id": 2,
                "name": "Neurology",
                "floor": 5
            },
            "exp": 8
        },
        {
            "id": 103,
            "name": "Dr. Anjali Silva",
            "spec": "Cardiologist",
            "department": {
                "id": 1,
                "name": "Cardiology",
                "floor": 3
            },
            "exp": 5
        }
    ],
    "appointments": [
        {
            "id": 1001,
            "patient": {
                "id": 501,
                "name": "Kamal Jayasinghe",
                "dob": "1990-04-15",
                "blood_type": "A+"
            },
            "doctor": {
                "id": 101,
                "name": "Dr. Sarah Fernando",
                "spec": "Cardiologist",
                "department": {
                    "id": 1,
                    "name": "Cardiology",
                    "floor": 3
                },
                "exp": 12
            },
            "sche": "2026-11-20T10:00:00",
            "dura": 30,
            "notes": "Follow-up consultation"
        },
        {
            "id": 1002,
            "patient": {
                "id": 502,
                "name": "Nadeesha Fernando",
                "dob": "1985-08-22",
                "blood_type": "O-"
            },
            "doctor": {
                "id": 102,
                "name": "Dr. Nimal Perera",
                "spec": "Neurologist",
                "department": {
                    "id": 2,
                    "name": "Neurology",
                    "floor": 5
                },
                "exp": 8
            },
            "sche": "2026-11-21T14:30:00",
            "dura": 45,
            "notes": None
        },
        {
            "id": 1003,
            "patient": {
                "id": 503,
                "name": "Ruwan Bandara",
                "dob": "1978-12-03",
                "blood_type": "B+"
            },
            "doctor": {
                "id": 103,
                "name": "Dr. Anjali Silva",
                "spec": "Cardiologist",
                "department": {
                    "id": 1,
                    "name": "Cardiology",
                    "floor": 3
                },
                "exp": 5
            },
            "sche": "2026-11-22T09:15:00",
            "dura": 20
        },
        {
            "id": 1004,
            "patient": {
                "id": 504,
                "name": "Samanthi Wijesinghe",
                "dob": "1995-06-30",
                "blood_type": "AB+"
            },
            "doctor": {
                "id": 101,
                "name": "Dr. Sarah Fernando",
                "spec": "Cardiologist",
                "department": {
                    "id": 1,
                    "name": "Cardiology",
                    "floor": 3
                },
                "exp": 12
            },
            "sche": "2026-11-23T11:00:00",
            "dura": 60,
            "notes": "ECG and blood pressure check"
        }
    ]
}


In [94]:

h1 = Hospital.model_validate(hospital_data)
h1.model_dump_json(exclude={"doctors", "departments"})

'{"name":"City General Hospital","appointments":[{"id":1001,"patient":{"id":501,"name":"Kamal Jayasinghe","dob":"1990-04-15","blood_type":"A+"},"doctor":{"id":101,"name":"Dr. Sarah Fernando","spec":"Cardiologist","department":{"id":1,"name":"Cardiology","floor":3},"exp":12},"sche":"2026-11-20T10:00:00","dura":30,"notes":"Follow-up consultation"},{"id":1002,"patient":{"id":502,"name":"Nadeesha Fernando","dob":"1985-08-22","blood_type":"O-"},"doctor":{"id":102,"name":"Dr. Nimal Perera","spec":"Neurologist","department":{"id":2,"name":"Neurology","floor":5},"exp":8},"sche":"2026-11-21T14:30:00","dura":45,"notes":null},{"id":1003,"patient":{"id":503,"name":"Ruwan Bandara","dob":"1978-12-03","blood_type":"B+"},"doctor":{"id":103,"name":"Dr. Anjali Silva","spec":"Cardiologist","department":{"id":1,"name":"Cardiology","floor":3},"exp":5},"sche":"2026-11-22T09:15:00","dura":20,"notes":null},{"id":1004,"patient":{"id":504,"name":"Samanthi Wijesinghe","dob":"1995-06-30","blood_type":"AB+"},"doct

In [92]:
def find_appointments(hospital: Hospital, doctor_id: int):
    appointment_list = []
    for appointment in hospital.appointments:
        if appointment.doctor.id is doctor_id:
            appointment_list.append(appointment)
    return appointment_list

print(find_appointments(hospital=h1, doctor_id=101))

[Appointment(id=1001, patient=Patient(id=501, name='Kamal Jayasinghe', dob=datetime.date(1990, 4, 15), blood_type='A+'), doctor=Doctor(id=101, name='Dr. Sarah Fernando', spec='Cardiologist', department=Department(id=1, name='Cardiology', floor=3), exp=12), sche=datetime.datetime(2026, 11, 20, 10, 0), dura=30, notes='Follow-up consultation'), Appointment(id=1004, patient=Patient(id=504, name='Samanthi Wijesinghe', dob=datetime.date(1995, 6, 30), blood_type='AB+'), doctor=Doctor(id=101, name='Dr. Sarah Fernando', spec='Cardiologist', department=Department(id=1, name='Cardiology', floor=3), exp=12), sche=datetime.datetime(2026, 11, 23, 11, 0), dura=60, notes='ECG and blood pressure check')]


In [10]:
from pydantic import ConfigDict, field_serializer, BaseModel, BeforeValidator, AfterValidator
import json

class RawAPIResponse(BaseModel):
    model_config = ConfigDict(
        frozen = True,
        str_strip_whitespace = True,
    )

    st_code: int
    body: str
    headers: dict[str, str]

class ParsedUser(BaseModel):
    id: int
    username: str
    email: str
    created_at: datetime
    role: Literal['admin','user','moderator']

    @field_serializer("created_at")
    def date_to_ISO(self, created_at: datetime, _info) -> datetime:
        return created_at.isoformat()

In [132]:
def process_response(raw: RawAPIResponse) -> list[ParsedUser]:
    p1 = [ParsedUser.model_validate(data) for data in  json.loads(raw.body)]
    return p1

In [113]:
users_json = """
[
  {
    "id": 1,
    "username": "  alice_admin  ",
    "email": "alice@example.com",
    "created_at": "2024-03-15T09:30:00",
    "role": "admin"
  },
  {
    "id": 2,
    "username": "bob_user",
    "email": "  bob@example.com ",
    "created_at": "2024-06-20T14:45:00",
    "role": "user"
  },
  {
    "id": 3,
    "username": "  charlie_mod  ",
    "email": "charlie@example.com",
    "created_at": "2025-01-10T11:00:00",
    "role": "moderator"
  },
  {
    "id": 4,
    "username": "diana",
    "email": "diana@example.com  ",
    "created_at": "2025-08-05T16:20:00",
    "role": "user"
  },
  {
    "id": 5,
    "username": "  eve_admin",
    "email": "  eve@example.com",
    "created_at": "2026-02-28T08:15:00",
    "role": "admin"
  }
]
"""

r1 = {
    "st_code": 200,
    "body": users_json,
    "headers": {
        "Content-Type": "application/json",
        "X-Request-ID": "abc-123"
    }
}

In [136]:
response = RawAPIResponse.model_validate(r1)
p1 = process_response(response)
p1[0].model_dump()

D:\Git\python_projects\python_projects\venv\Lib\site-packages\pydantic\main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `datetime` - serialized value may not be as expected [field_name='created_at', input_value='2024-03-15T09:30:00', input_type=str])
  return self.__pydantic_serializer__.to_python(


{'id': 1,
 'username': '  alice_admin  ',
 'email': 'alice@example.com',
 'created_at': '2024-03-15T09:30:00',
 'role': 'admin'}

In [11]:
def verify_currency(currency: str) -> str:
    if not isinstance(currency, str):
        raise ValueError("Currency should be a string")
    cleaned = currency.strip().upper()

    if len(cleaned) != 3 or not cleaned.isalpha():
        raise ValueError("Currency must be exactly 3 uppercase letters")
    return cleaned

def verify_amount(ammount: float) -> float:
    ammount_cleaned = round(ammount, 2)
    if ammount_cleaned < 0:
        raise ValueError("Ammount must be greater than or equal 0")
    return ammount_cleaned

def IBAN_strip(IBAN: str) -> str:
    return IBAN.strip()

def IBAN_verify(IBAN: str) -> str:
    if not IBAN.isalnum():
        raise ValueError("IBAN invalid!")
    if not IBAN[:2].isalpha() or not IBAN[2:4].isdigit() or len(IBAN) < 30:
        raise ValueError("IBAN invalid!")
    return IBAN

Currency = Annotated[str, BeforeValidator(verify_currency)]
Ammount = Annotated[float, AfterValidator(verify_amount)]
IBAN = Annotated[str, BeforeValidator(IBAN_strip), AfterValidator(IBAN_verify)]
PositiveInt = Annotated[int, Field(ge=0)]

In [26]:
from typing import Optional, Self

class Bank(BaseModel):
    transfer_id: PositiveInt
    from_iban: IBAN
    to_iban: IBAN
    amount: Ammount
    description: Optional[str] = None
    currency: Currency

    @model_validator(mode="after")
    def verify_iban_equal(self) -> Self:
        if self.from_iban == self.to_iban:
            raise ValueError("Trannsfer account and reciever account same!!")
        return self

In [20]:
class Portfolio(BaseModel):
    owner: str
    holdings: dict[Currency, Ammount]

    @property
    def num_holdings(self):
        return len(self.holdings)

In [27]:
transfers = [
    Bank(
        transfer_id=1001,
        from_iban="  GB82WEST1234569876543200000001  ",
        to_iban="DE8937040044053201300000000002",
        amount=1250.50,
        currency=" eur ",
        description="Invoice #409 payment",
    ),
    Bank(
        transfer_id=1002,
        from_iban="FR7630006000011234567890189003",
        to_iban="ES9121000418450200051332000004",
        amount=499.999,  # Rounds to 500.0
        currency="USD",
        description="Software subscription",
    ),
    Bank(
        transfer_id=1003,
        from_iban="NL91ABNA0417164300000000000005",
        to_iban="CH9300762011623852957000000006",
        amount=8750.00,
        currency="chf",
        # description omitted (defaults to None)
    ),
    Bank(
        transfer_id=1004,
        from_iban="IT60X0542811101000000123456007",
        to_iban="BE6853900754703400000000000008",
        amount=0.0,
        currency="GBP",
        description="Account verification micro-transfer",
    ),
    Bank(
        transfer_id=1005,
        from_iban="AE0703312345678901234560000009",
        to_iban="SA0380000000608010167519000010",
        amount=3420.75,
        currency="AED",
        description="Consulting retainer",
    ),
]

In [50]:
b_list = []
def model_val(transfers):    
    for transfer in transfers:
        b_list.append(Bank.model_validate(transfer))
model_val(transfers)
b_list[0].model_dump()

{'transfer_id': 1001,
 'from_iban': 'GB82WEST1234569876543200000001',
 'to_iban': 'DE8937040044053201300000000002',
 'amount': 1250.5,
 'description': 'Invoice #409 payment',
 'currency': 'EUR'}

In [51]:
from pydantic_settings import BaseSettings, SettingsConfigDict

class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file = ".env",
        env_file_encoding="utf-8",
        case_sensitive=False, 
    )

    app_name: str = "MyApp"
    db_url: str 
    sc_key: str